# ZuCo Task1-SR — Spatial Sentiment Decoding (thin Colab driver)

The pipeline code lives in the **eeg-spatial-encoding** GitHub repo. This notebook just
clones/updates that repo and runs it, so the notebook itself rarely needs to change — to
pick up code changes, re-run the **Setup** cell (it does a `git pull`).


## Setup — pull latest code from GitHub + install deps


In [ ]:
# Re-run this cell any time to pick up the latest pushed code.
import os, sys

REPO_URL = 'https://github.com/parmisbathaeiyan/eeg-spatial-encoding.git'
REPO_DIR = '/content/eeg-spatial-encoding'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

!pip install -q -r {REPO_DIR}/requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('code ready:', REPO_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Config — set your Drive paths


In [ ]:
from eeg_spatial.config import Config

# All data (mat files, label CSV, montage) lives on your Drive - verify these paths.
cfg = Config(
    sr_data_dir='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_og_raw',
    labels_csv='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_sentiment_labels_task1_fixed.csv',
    montage_npz='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_montage.npz',
    feature_prefix='TRT',   # FFD | TRT | GD | GPT
    k_neighbors=4,
)

# where to save per-electrode accuracy results
RESULTS_CSV = '/content/drive/MyDrive/Parmis/Thesis/Results/spatial_accuracy/spatial_accuracy_results.csv'


## Build dataset + montage + neighbor graph


In [ ]:
from eeg_spatial import dataset, montage

label_map = dataset.load_label_map(cfg.labels_csv)
X_band, y, meta = dataset.build_band_dataset(cfg, label_map)

ch_names, ch_pos, mont = montage.load_montage(cfg.montage_npz)
neighbor_idx = montage.build_neighbor_graph(ch_pos, cfg.k_neighbors)
info = montage.make_info(ch_names, mont)
assert X_band.shape[1] == len(ch_names), 'channel-count mismatch between data and montage'


## Sentence-level searchlight (band power averaged over words) + topomap


In [ ]:
from eeg_spatial import searchlight, plotting

results_df = searchlight.run_searchlight(
    X_band, y, neighbor_idx, ch_names, n_folds=cfg.n_folds, random_state=cfg.random_state)
searchlight.report_searchlight(results_df)

os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
results_df.to_csv(RESULTS_CSV)
print('saved ->', RESULTS_CSV)

plotting.plot_accuracy_topomaps(results_df, info, cfg.k_neighbors)


## (Optional) Word-concatenated features (no averaging)

Keep every word's (105 x 8) vector, padded/truncated to a fixed word count, instead of
averaging. Same searchlight, larger per-electrode feature vector.


In [ ]:
X_concat, y_concat, max_words = dataset.build_concat_dataset(cfg, label_map, max_words='p95')
results_concat = searchlight.run_searchlight(
    X_concat, y_concat, neighbor_idx, ch_names, n_folds=cfg.n_folds, random_state=cfg.random_state)
searchlight.report_searchlight(results_concat)
results_concat.to_csv(RESULTS_CSV.replace('.csv', '_wordconcat.csv'))
plotting.plot_accuracy_topomaps(results_concat, info, cfg.k_neighbors, title_suffix=' (concatenated words)')


## (Optional) 1D-CNN searchlight on raw EEG

Heavy: rebuilds the dataset keeping raw EEG, then trains one CNN per electrode
neighborhood. **Use a GPU runtime** (Runtime -> Change runtime type -> GPU).


In [ ]:
import torch
from eeg_spatial import cnn

cfg.keep_raw_eeg = True
X_band, y, meta = dataset.build_band_dataset(cfg, label_map)   # rebuild with raw EEG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
cnn_acc = cnn.run_cnn_searchlight(
    meta['raw_eeg'], y, neighbor_idx, ch_names, device,
    epochs=15, batch_size=16, n_folds=cfg.n_folds, random_state=cfg.random_state)

results_df['CNN'] = cnn_acc
results_df.to_csv(RESULTS_CSV)
plotting.plot_accuracy_topomaps(results_df, info, cfg.k_neighbors, title_suffix=' (incl. CNN)')
